# 04 ML Classical Algorithms — Baseline

**Baseline group.** Trains regressors using only classical network features (DebtRank, PageRank, centrality measures) with no embedding input. Used as the reference point for all embedding experiments.

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

sys.path.insert(0, os.path.abspath('../..'))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_classical_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

## Load Dataset

In [2]:
print(PROJECT_ROOT)

C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


In [3]:
df, feature_cols = load_classical_dataset(PROJECT_ROOT)
df.shape, feature_cols

((145536, 90),
 ['degree_centrality_total',
  'weighted_degree_total',
  'betweenness_centrality',
  'closeness_centrality',
  'eigenvector_centrality',
  'pagerank',
  'debtrank'])

In [4]:
trainer = ModelTrainer(
    df=df,
    feature_cols=feature_cols,
    target_col="log_systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 90), (18192, 90), (13644, 90))

## Define Models

In [ ]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(32,63,16), max_iter=300, activation='relu', learning_rate="adaptive", early_stopping=True, n_iter_no_change=10, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

## Train And Store

In [6]:
trainer.train_all(candidate_models)
trainer.leaderboard()[DISPLAY_COLS]

c:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.9798088627289653e-77.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.003502,0.014545,0.021117,0.08289
1,Gradient Boosting,0.009013,0.014706,0.053364,0.083878
2,XGBoost,0.00797,0.015085,0.044758,0.084803
3,Ridge,0.024997,0.037311,0.095156,0.150539
4,Linear Regression,0.040509,0.051685,0.176157,0.23651
5,MLP,45479499213914985950347264.0,9858213852915.951172,8206783496658376125027713024.0,993024298422562.125


## Hyperparameter Tuning

Uses `RandomizedSearchCV` with `PredefinedSplit` so the temporal train/val boundary is respected — train rows are always used for fitting, val rows always for scoring.

In [8]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model,
        param_distributions,
        n_iter=n_iter,
        cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    return search.best_params_

RF_PARAMS = {
    "model__n_estimators":     randint(100, 600),
    "model__max_depth":        [None, 5, 10, 15, 20, 30],
    "model__min_samples_leaf": randint(1, 20),
    "model__min_samples_split":randint(2, 20),
    "model__max_features":     ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          randint(100, 600),
    "model__max_depth":         [3, 4, 5, 6, 8, None],
    "model__learning_rate":     loguniform(0.005, 0.3),
    "model__min_samples_leaf":  randint(5, 100),
    "model__l2_regularization": loguniform(1e-4, 1.0),
    "model__max_leaf_nodes":    randint(15, 60),
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     randint(100, 800),
    "model__max_depth":        randint(3, 10),
    "model__learning_rate":    loguniform(0.005, 0.3),
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": randint(1, 10),
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        loguniform(1e-5, 1.0),
    "model__reg_lambda":       loguniform(1e-5, 2.0),
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":               loguniform(1e-5, 0.1),
    "model__learning_rate_init":  loguniform(1e-4, 0.05),
    "model__learning_rate":       ["constant", "adaptive"],
    "model__batch_size":          [32, 64, 128, "auto"],
}

In [9]:
tune(trainer, make_pipeline(RandomForestRegressor(random_state=42)),     RF_PARAMS,  "Random Forest (tuned)")
tune(trainer, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS, "Gradient Boosting (tuned)")
tune(trainer, make_pipeline(XGBRegressor(random_state=42)),              XGB_PARAMS, "XGBoost (tuned)")
tune(trainer, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, "MLP (tuned)")
trainer.leaderboard()[DISPLAY_COLS]

Tuning complete


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.006986,0.014108,0.041251,0.07859
1,Gradient Boosting (tuned),0.00979,0.014599,0.057698,0.081501
2,Random Forest,0.003502,0.014545,0.021117,0.08289
3,Gradient Boosting,0.009013,0.014706,0.053364,0.083878
4,XGBoost,0.00797,0.015085,0.044758,0.084803
5,XGBoost (tuned),0.007415,0.015556,0.039703,0.085066
6,Ridge,0.024997,0.037311,0.095156,0.150539
7,MLP (tuned),0.030189,0.038914,0.148088,0.198254
8,Linear Regression,0.040509,0.051685,0.176157,0.23651
9,MLP,45479499213914985950347264.0,9858213852915.951172,8206783496658376125027713024.0,993024298422562.125


In [ ]:
trainer.leaderboard()[TOP1_COLS]